In [42]:
MANUSCRIPTS = ["P5881", "P400", "CCCP578", "P3465", "P3466"]
CHAPTERS = ['Lv', 'Im', 'Oc', 'Mc', 'Kd', 'Km','Ag','Lj','Di']

In [43]:
import json
import os
with open("manuscripts.json", "r", encoding="utf-8") as file:
    manuscripts_src = json.loads(file.read())

with open("units.json", "r", encoding="utf-8") as file:
    units_src = json.loads(file.read())
with open("segments.json", "r", encoding="utf-8") as file:
    segments_src = json.loads(file.read())
with open("pages.json", "r", encoding="utf-8") as file:
    pages_src = json.loads(file.read())
with open("text.json", "r", encoding="utf-8") as file:
    text_src = json.loads(file.read())
with open("lines.json", "r", encoding="utf-8") as file:
    lines_src = json.loads(file.read())
with open("images.json", "r", encoding="utf-8") as file:
    images_src = json.loads(file.read())

In [44]:
def get_mss_siglum_to_id(ms_list):
    res = {}
    for siglum in ms_list:
        for ms in manuscripts_src:
            if siglum == ms['siglum']:
                res[siglum] = ms['id']
    return res
def get_units(chapter):
    units = [x for x in units_src if x.get('frame', '') == chapter]
    unit_ids = {x['id'] for x in units}
    for unit in units:
        unit['longestSegment'] = {'count': 0, 'siglum': ''}
    return units, unit_ids
def format_units(units, root_unit_title):
    root_unit = next((x for x in units if x['title'] == root_unit_title), None)
    root_unit['formattedOrder'] = root_unit['frame']
    def get_childern(root_id, order):
        children = []
        for unit in units:
            if unit['parentId'] == root_id:
                unit['formattedOrder'] = f"{root_unit['frame']}.{unit['order']}"
                children.append(unit)
                children += get_childern(unit['id'], unit['order'])
        return children
    return root_unit, get_childern(root_unit['id'], f"{root_unit['order']}")
def get_segments(ms_id, unit_ids):
    segments = [x for x in segments_src if x['unitId'] in unit_ids and x['mediumId'] == ms_id]
    return {x['unitId']: x for x in segments}
def get_segment_pages(ms_id, segment):
    pages = [x for x in pages_src if x['mediumId'] == ms_id and x['number']>= segment['startPage'] and x['number'] <= segment['endPage']  ]
    return pages

def get_segment_bounding_pages(pages, segment):
    first = next((x for x in pages if x['number'] == segment['startPage']), None)
    last = next((x for x in pages if x['number'] == segment['endPage']), None)
    return first, last
def get_page_body_lines(page):
    element_ids = {x['id'] for x in text_src if x['pageId'] == page['id'] and 'main' in x['position']}
    lines = [x for x in lines_src if x['elementId'] in element_ids]
    for line in lines:
        line['pageNumber'] = page['number']
        line['pageId'] = page['id']
    lines.sort(key=lambda item: item['order'])
    return lines

def get_segment_lines(lines, segment, start_page_id, end_page_id):
    if segment["startPage"] == segment["endPage"]:
        return [
            x
            for x in lines
            if x["order"] >= segment["startLine"] and x["order"] <= segment["endLine"]
        ]
    if segment["endPage"] - segment["startPage"] == 1:
        lines_from_start_page = [
            x
            for x in lines
            if x["order"] >= segment["startLine"] and x["pageId"] == start_page_id
        ]
        lines_from_start_page.sort(key=lambda item: item["order"])
        lines_from_end_page = [
            x
            for x in lines
            if x["order"] <= segment["endLine"] and x["pageId"] == end_page_id
        ]
        lines_from_end_page.sort(key=lambda item: item["order"])
        return lines_from_start_page + lines_from_end_page
    if segment["endPage"] - segment["startPage"] > 1:
        lines_from_start_page = [
            x
            for x in lines
            if x["order"] >= segment["startLine"] and x["pageId"] == start_page_id
        ]
        lines_from_start_page.sort(key=lambda item: item["order"])
        lines_from_mid_pages = [
            x
            for x in lines
            if x["pageId"] != start_page_id and  x["pageId"] != end_page_id
        ]
        lines_from_mid_pages.sort(key=lambda item: (item["order"], item['pageNumber']))
        lines_from_end_page = [
            x
            for x in lines
            if x["order"] <= segment["endLine"] and x["pageId"] == end_page_id
        ]
        lines_from_end_page.sort(key=lambda item: item["order"])
        return lines_from_start_page + lines_from_mid_pages + lines_from_end_page

def get_line_data(line, start=None, end=None):
    if start is not None and end is not None:
        tokens = line['tokens'][start:end]
        states = line['states'][start:end]
    elif start is not None:
        tokens = line['tokens'][start:]
        states = line['states'][start:]
    elif end is not None:
        tokens = line['tokens'][:end]
        states = line['states'][:end]
    else:
        tokens = line['tokens']
        states = line['states']

    return {
        'tokens': tokens,
        'states': states,
        'lines': [line['order']] * len(tokens),
        'pages': [line['pageNumber']] * len(tokens),
        'breaks': [None if i != 0 or line['order'] != 0 else line['pageNumber']
                   for i in range(len(tokens))]
    }

def get_segment_data_same_line(lines, segment):
    line_data = get_line_data(lines[0], start=segment['startToken'], end=segment['endToken']+1)
    data = {
        'images': [],
        'lemmas': []
    }
    data.update(line_data)
    return data



def get_segment_data_same_page(lines, segment):
    data = {
        'tokens': [],
        'states': [],
        'lines': [],
        'pages': [],
        'images': [],
        'breaks': [],
        'lemmas': []
    }
    for line in lines:
        if line['order'] == segment['startLine']:
            line_data = get_line_data(line, start=segment['startToken'])
        elif line['order'] == segment['endLine']:
            line_data = get_line_data(line, end=segment['endToken']+1)
        else:
            line_data = get_line_data(line)

        data['tokens'] += line_data['tokens']
        data['states'] += line_data['states']
        data['lines'] += line_data['lines']
        data['pages'] += line_data['pages']
        data['breaks'] += line_data['breaks']

    return data


def get_segment_data_multiple_pages(lines, segment):
    data = {
        'tokens': [],
        'states': [],
        'lines': [],
        'pages': [],
        'images': [],
        'breaks': [],
        'lemmas': []
    }
    for line in lines:
        if line['order'] == segment['startLine'] and line['pageNumber'] == segment['startPage']:
            line_data = get_line_data(line, start=segment['startToken'])
        elif line['order'] == segment['endLine'] and line['pageNumber'] == segment['endPage']:
            line_data = get_line_data(line, end=segment['endToken']+1)
        else:
            line_data = get_line_data(line)

        data['tokens'] += line_data['tokens']
        data['states'] += line_data['states']
        data['lines'] += line_data['lines']
        data['pages'] += line_data['pages']
        data['breaks'] += line_data['breaks']

    return data


def get_segment_data(lines, segment):
    if segment['startPage'] == segment['endPage']:
        if segment['startLine'] == segment['endLine']:
            return get_segment_data_same_line(lines, segment)
        else:
            return get_segment_data_same_page(lines, segment)
    else:
        return get_segment_data_multiple_pages(lines, segment)


In [45]:
# chapter_units
for chapter in CHAPTERS:
  units, unit_ids = get_units(chapter)
  units.sort(key=lambda item: item["order"])
  units = [{'title': x['title'], 'number':x['order'] } for x in units]
  out_file = f"../apps/data-api/data/manuscripts/chapter_units/{chapter}.json"
  with open(out_file, "w", encoding='utf-8') as write_file:
    json.dump(units, write_file, indent=4)


In [46]:
# chapter_to_ms
mss_siglum_to_id = get_mss_siglum_to_id(MANUSCRIPTS)

for chapter in CHAPTERS:
  units, unit_ids = get_units(chapter)
  units.sort(key=lambda item: item["order"])
  units = [{'title': x['title'], 'number':x['order'] } for x in units]
  data =[]
  for i, ms in enumerate(MANUSCRIPTS):
    id = mss_siglum_to_id[ms]
    segments = get_segments(id, unit_ids).values()
    if len(segments) != 0:
      start_page = min([x['startPage'] for x in segments])
      end_page = max([x['endPage'] for x in segments])
      data.append({"manuscript": ms, 'from': start_page, 'to': end_page})

  out_file = f"../apps/data-api/data/manuscripts/test/{chapter}.json"
  with open(out_file, "w", encoding='utf-8') as write_file:
    json.dump(data, write_file, indent=4)



FileNotFoundError: [Errno 2] No such file or directory: '../apps/data-api/data/manuscripts/test/Lv.json'

In [209]:
print(lines_src[20])


{'elementId': '2ba2d549-7c6a-4994-bfab-d49080c398d2', 'id': '8af960af-bbb5-41d7-a7d6-2b1061f419e8', 'order': 3, 'region': [134, 875, 1724, 875, 1724, 1031, 134, 1031, 0], 'tokens': ['الشأن', 'فوُصف', 'للوزير', 'فوجّه', 'إليه', 'فأحضره', 'وناظره', 'على', 'أمر'], 'states': ['sound', 'sound', 'sound', 'sound', 'sound', 'sound', 'sound', 'sound', 'sound']}


In [41]:
# The error you are seeing is:
# OpenCV(4.5.2) ../modules/imgproc/src/imgwarp.cpp:3144: error: (-215:Assertion failed) _src.total() > 0 in function 'warpPerspective'
# This means that the image you are trying to process with cv2.warpPerspective is empty or not loaded correctly.
# 
# Most likely, the file path you are passing to cv2.imread does not exist, or the image file is missing or corrupted.
# 
# To fix this:
# 1. Make sure you have OpenCV installed: `pip install opencv-python`
# 2. Make sure the image file exists at the path you are passing to cv2.imread.
# 3. You can add a check after loading the image to see if it loaded correctly, and print a helpful error message if not.

import cv2
from PIL import Image
import numpy as np

# Create a dictionary to store the illustrations for each chapter
illustration_data = {}

for ms_siglum in MANUSCRIPTS:
    for chapter in CHAPTERS:

        ms_id = mss_siglum_to_id[ms_siglum]
        units, unit_ids = get_units(chapter)
        segments = get_segments(ms_id, unit_ids)

        if segments:  # Check if segments is not empty
            for page_number in range(min([x['startPage'] for x in segments.values()]), max([x['endPage'] for x in segments.values()])+1):
                page_segments = [x for x in segments.values() if x['startPage'] <= page_number <= x['endPage']]
                page_units = [x for x in units if x['id'] in [y['unitId'] for y in page_segments]]
                page_data = {
                    "id": "",
                    "number": page_number,
                    "manuscript": ms_siglum,
                    "imageUrl": "",
                    "lines": [],
                    "unitPlaces": [],
                    "unitNames": [],
                    "illustrationUrls": [],
                    "region_of_illustration": [],
                    "position_of_illustration": [],
                    "location": []
                }

                # Check for units from other chapters
                for other_chapter in CHAPTERS:
                    if other_chapter != chapter:
                        other_units, other_unit_ids = get_units(other_chapter)
                        other_segments = get_segments(ms_id, other_unit_ids)
                        other_page_segments = [x for x in other_segments.values() if x['startPage'] <= page_number <= x['endPage']]
                        other_page_units = [x for x in other_units if x['id'] in [y['unitId'] for y in other_page_segments]]
                        for unit in other_page_units:
                            segment = next((x for x in other_page_segments if x['unitId'] == unit['id']), None)
                            if segment is not None:
                                if segment['startPage'] == page_number:
                                    page_data['unitPlaces'].append([segment['startLine'], segment['startToken']])
                                    page_data['unitNames'].append([other_chapter + ' ' + unit['title'], unit['order']])

                for unit in page_units:
                    segment = next((x for x in page_segments if x['unitId'] == unit['id']), None)
                    if segment is not None:
                        if segment['startPage'] == page_number:
                            page_data['unitPlaces'].append([segment['startLine'], segment['startToken']])
                            page_data['unitNames'].append([chapter + ' ' + unit['title'], unit['order']])
                for segment in page_segments:
                    pages = get_segment_pages(ms_id, segment)
                    for page in pages:
                        if page['number'] == page_number:
                            page_data['imageUrl'] = page['image']
                            page_data['id'] = page['id']
                            lines = get_page_body_lines(page)
                            for line in lines:
                                if line['tokens'] not in page_data['lines']:
                                    page_data['lines'].append(line['tokens'])

                for image in images_src:
                    if image['pageId'] == page_data['id']:
                        page_data['region_of_illustration'].append(image['region'])
                        page_data['position_of_illustration'].append(image['position'])
                        page_data['location'].append(image['location'])

                        # Load the image
                        image_path = f"../apps/data-api/images/pages/{page_data['imageUrl']}"
                        img = cv2.imread(image_path)

                        if img is None or img.size == 0:
                            print(f"ERROR: Could not load image at {image_path}. Please check that the file exists and is a valid image.")
                            continue  # Skip this illustration

                        # Get the region of the illustration
                        region = image['region']
                        x1, y1, x2, y2, x3, y3, x4, y4, r = region
                        points = np.array([[x1, y1], [x2, y2], [x3, y3], [x4, y4]], dtype=np.float32)
                        width = x2 - x1
                        height = y3 - y1
                        new_points = np.array([[0, 0], [width - 1, 0], [width - 1, height - 1], [0, height - 1]], dtype=np.float32)

                        matrix = cv2.getPerspectiveTransform(points, new_points)
                        cropped_img = cv2.warpPerspective(img, matrix, (width, height))

                        # Save the cropped image
                        directory = f"../apps/data-api/data/manuscripts/test/illustrations/all_iillustrations"
                        os.makedirs(directory, exist_ok=True)
                        out_file = f"{directory}/illustration_{page_data['number']}_{image['id']}.jpg"
                        if cropped_img is not None and cropped_img.size > 0:
                            cv2.imwrite(out_file, cropped_img)
                            page_data['illustrationUrls'].append(f"illustration_{page_data['number']}_{image['id']}.jpg")
                            if chapter not in illustration_data:
                                illustration_data[chapter] = {}
                            if ms_siglum not in illustration_data[chapter]:
                                illustration_data[chapter][ms_siglum] = []
                            illustration_data[chapter][ms_siglum].extend(page_data['illustrationUrls'])
                        else:
                            print(f"ERROR: Cropped image is empty for {out_file}")

                directory = f"../apps/data-api/data/manuscripts/test/{ms_siglum}/{chapter}"
                os.makedirs(directory, exist_ok=True)
                out_file = f"{directory}/{page_number}.json"
                with open(out_file, "w", encoding='utf-8') as write_file:
                    json.dump(page_data, write_file, indent=4, ensure_ascii=False)

organized_illustration_data = []

for chapter in illustration_data:
    chapter_data = {
        "chapter": chapter, "manuscripts": []
    }

    for manuscript in illustration_data[chapter]:
        manuscript_data = {
            "siglum": manuscript,
            "illustrations": illustration_data[chapter][manuscript]
        }
        chapter_data["manuscripts"].append(manuscript_data)

    organized_illustration_data.append(chapter_data)
    # Collect the illustrations for each chapter in one json file
    illustration_file = "../apps/data-api/data/manuscripts/test/illustration_data.json"
    with open(illustration_file, "w", encoding='utf-8') as write_file:
        json.dump(illustration_data, write_file, indent=4, ensure_ascii=False)

illustration_file_organized = "../apps/data-api/data/manuscripts/test/illustration_data_organized.json"
with open(illustration_file_organized, "w", encoding='utf-8') as write_file:
    json.dump(organized_illustration_data, write_file, indent=4, ensure_ascii=False)


ERROR: Could not load image at ../apps/data-api/images/pages/35229c64-f47b-4384-895b-504106d06b79.jpg. Please check that the file exists and is a valid image.
ERROR: Could not load image at ../apps/data-api/images/pages/97923ee3-de00-4919-8044-011e4777dd58.jpg. Please check that the file exists and is a valid image.
ERROR: Could not load image at ../apps/data-api/images/pages/39dd39c3-3665-4b49-9d3a-073e7949d8b3.jpg. Please check that the file exists and is a valid image.
ERROR: Could not load image at ../apps/data-api/images/pages/a56b1081-b3ab-48eb-9444-003eaa0d8f77.jpg. Please check that the file exists and is a valid image.
ERROR: Could not load image at ../apps/data-api/images/pages/c0454d27-a8a3-4ac1-b2e2-75e2181c2032.jpg. Please check that the file exists and is a valid image.
ERROR: Could not load image at ../apps/data-api/images/pages/bacb4da9-0139-43d3-b98d-120d5a78ce69.jpg. Please check that the file exists and is a valid image.
ERROR: Could not load image at ../apps/data-ap

In [9]:
#all the pages in one manuscript
for ms_siglum in MANUSCRIPTS:
    all_pages = []
    for chapter in CHAPTERS:

        ms_id = mss_siglum_to_id[ms_siglum]
        units, unit_ids = get_units(chapter)
        segments = get_segments(ms_id, unit_ids)

        if segments:  # Check if segments is not empty
            for page_number in range(min([x['startPage'] for x in segments.values()]), max([x['endPage'] for x in segments.values()])+1):
                page_segments = [x for x in segments.values() if x['startPage'] <= page_number <= x['endPage']]
                page_units = [x for x in units if x['id'] in [y['unitId'] for y in page_segments]]
                page_data = {
                    "index": page_number,
                    "page_number": page_number,
                    "page_link": f"/manuscripts/{ms_siglum}/{chapter}/{page_number}"
                }
                # Check if page_number already exists in all_pages
                if page_number not in [page['page_number'] for page in all_pages]:
                    all_pages.append(page_data)

    # Sort all_pages by index before writing to file
    all_pages.sort(key=lambda page: page['index'])

    directory = f"../apps/data-api/data/manuscripts/test/{ms_siglum}"
    os.makedirs(directory, exist_ok=True)
    out_file = f"{directory}/allPages.json"
    with open(out_file, "w", encoding='utf-8') as write_file:
        json.dump(all_pages, write_file, indent=4, ensure_ascii=False)



In [10]:
# the pages in one chapter
for ms_siglum in MANUSCRIPTS:
    ms_id = mss_siglum_to_id[ms_siglum]
    for chapter in CHAPTERS:
        units, unit_ids = get_units(chapter)
        segments = get_segments(ms_id, unit_ids)
        pages_in_chapter = []

        if segments:  # Check if segments is not empty
            for page_number in range(min([x['startPage'] for x in segments.values()]), max([x['endPage'] for x in segments.values()])+1):
                page_segments = [x for x in segments.values() if x['startPage'] <= page_number <= x['endPage']]
                page_units = [x for x in units if x['id'] in [y['unitId'] for y in page_segments]]
                page_data = {
                    "index": page_number,
                    "page_number": page_number,
                    "page_link": f"/manuscripts/{ms_siglum}/{chapter}/{page_number}"
                }
                pages_in_chapter.append(page_data)

        directory = f"../apps/data-api/data/manuscripts/test/{ms_siglum}/{chapter}"
        os.makedirs(directory, exist_ok=True)
        out_file = f"{directory}/pagesInTheChapter.json"
        with open(out_file, "w", encoding='utf-8') as write_file:
            json.dump(pages_in_chapter, write_file, indent=4, ensure_ascii=False)


In [11]:
for ms_siglum in MANUSCRIPTS:
    all_chapters = []
    last_page_number = 0
    for chapter in CHAPTERS:

        ms_id = mss_siglum_to_id[ms_siglum]
        units, unit_ids = get_units(chapter)
        segments = get_segments(ms_id, unit_ids)
        all_pages = []

        if segments:  # Check if segments is not empty
            for page_number in range(min([x['startPage'] for x in segments.values()]), max([x['endPage'] for x in segments.values()])+1):
                page_segments = [x for x in segments.values() if x['startPage'] <= page_number <= x['endPage']]
                page_units = [x for x in units if x['id'] in [y['unitId'] for y in page_segments]]
                page_data = {
                    "index": page_number,
                    "page_number": page_number,
                    "page_link": f"/manuscripts/{ms_siglum}/{chapter}/{page_number}"
                }
                all_pages.append(page_data)

        # Sort all_pages by page_number before adding to chapter_data
        all_pages.sort(key=lambda page: page['page_number'])

        chapter_data = {
            "chapter": chapter,
            "pages": all_pages
        }

        # Check if the first page number of the current chapter is smaller than the last page number of the last chapter
        if all_pages and all_pages[0]['page_number'] < last_page_number:
            # If so, insert the current chapter at the correct position
            for i, ch in enumerate(all_chapters):
                if ch['pages'] and all_pages[0]['page_number'] < ch['pages'][0]['page_number']:
                    all_chapters.insert(i, chapter_data)
                    break
        else:
            all_chapters.append(chapter_data)

        # Update the last page number
        if all_pages:
            last_page_number = all_pages[-1]['page_number']

    directory = f"../apps/data-api/data/manuscripts/test/{ms_siglum}"
    os.makedirs(directory, exist_ok=True)
    out_file = f"{directory}/allChapters.json"
    with open(out_file, "w", encoding='utf-8') as write_file:
        json.dump(all_chapters, write_file, indent=4, ensure_ascii=False)




In [34]:
for ms_siglum in MANUSCRIPTS:
    gallery = []
    added_pages = set()  # Set to keep track of added pages
    for chapter in CHAPTERS:

        ms_id = mss_siglum_to_id[ms_siglum]
        units, unit_ids = get_units(chapter)
        segments = get_segments(ms_id, unit_ids)

        if segments:  # Check if segments is not empty
            for page_number in range(min([x['startPage'] for x in segments.values()]), max([x['endPage'] for x in segments.values()])+1):
                page_segments = [x for x in segments.values() if x['startPage'] <= page_number <= x['endPage']]
                page_units = [x for x in units if x['id'] in [y['unitId'] for y in page_segments]]
                page_data = {
                    "index": page_number,
                    "page_number": page_number,
                    "page_link": f"/manuscripts/{ms_siglum}/{chapter}/{page_number}"
                }
                for segment in page_segments:
                    pages = get_segment_pages(ms_id, segment)
                    for page in pages:
                        if page['number'] == page_number and page['id'] not in added_pages:  # Check if page has already been added
                            page_data['imageUrl']=page['image']
                            page_data['id']=page['id']
                            gallery.append({
                                "caption": f"{chapter} {page_number}",
                                "src": page_data['imageUrl'],
                                "thumb": page_data['imageUrl'],
                                "subHtml": f"{chapter} {page_number}",
                                "pageNumber": page_number  # Add page number for sorting
                            })
                            added_pages.add(page['id'])  # Add page id to the set of added pages

    # Sort the gallery by page_number before writing to file
    gallery.sort(key=lambda item: item['pageNumber'])

    directory = f"../apps/data-api/data/manuscripts/test/{ms_siglum}"
    os.makedirs(directory, exist_ok=True)
    out_file = f"{directory}/gallery.json"
    with open(out_file, "w", encoding='utf-8') as write_file:
        json.dump(gallery, write_file, indent=4, ensure_ascii=False)




In [ ]:
import json

def add_lines_to_list(lines_list, new_lines):
    # Split each line of text into words and append to the provided lines_list
    for line in new_lines:
        words = line.split()
        lines_list.append(words)

def save_lines_to_json(lines_list, json_filename):
    # Save the lines list to a JSON file
    with open("../apps/data-api/data/manuscripts/test/English/en.json",'w') as json_file:
        json.dump(lines_list, json_file)

# Initialize the "lines" list
lines = []

# Add lines to the "lines" list using the function
lines_to_add = []

add_lines_to_list(lines, lines_to_add)

# Save the lines list to a JSON file
json_filename = "lines.json"
save_lines_to_json(lines, json_filename)



In [2]:
import json
import re
import os
import pandas as pd
import xlsxwriter

# Read JSON data from file
def read_json(filename):
    with open(filename, 'r', encoding='utf-8') as file:
        return json.load(file)

# Replace problematic characters in the JSON data
# Remove HTML tags from the JSON data
def clean_json_data(data_list):
    cleaned_data = []
    for data in data_list:
        cleaned_entry = {}
        for key, value in data.items():
            if isinstance(value, str):
                cleaned_value = value.encode('utf-8', 'replace').decode('utf-8')
                cleaned_value = re.sub(r'<i>(.*?)</i>', r'\1', cleaned_value)
                # Convert <a> tags into hyperlinks
                cleaned_value = re.sub(r'<a href="(.*?)">(.*?)</a>', r'=HYPERLINK("\1", "\2")', cleaned_value)

                # Remove all HTML tags
                cleaned_value = re.sub(r'<.*?>', '', cleaned_value)
                cleaned_entry[key] = cleaned_value
            else:
                cleaned_entry[key] = value
        cleaned_data.append(cleaned_entry)
    return cleaned_data


# Convert JSON data to Excel
def json_to_excel(data_list, output_filename):
    # Convert the list of dictionaries to a DataFrame
    df = pd.DataFrame(data_list)

    # Replace underscores and double underscores in column names with spaces
    df.columns = df.columns.str.replace('_', ' ').str.replace('__', ' ')

    # Define ExcelWriter object and set dimensions
    with pd.ExcelWriter(output_filename, engine='xlsxwriter') as writer:
        df.to_excel(writer, index=False, encoding='utf-8-sig')

        # Get the xlsxwriter workbook and worksheet objects
        workbook  = writer.book
        worksheet = writer.sheets['Sheet1']  # Assuming there is only one sheet

        # Iterate through each column and set the width based on the text length
        for col_num, column in enumerate(df.columns):
            max_length = df[column].astype(str).map(len).max()
            if max_length > 50:
                worksheet.set_column(col_num, col_num, 50)
            else:
                worksheet.set_column(col_num, col_num, 10)

        # Enable text wrapping for all cells
        cell_format = workbook.add_format({'text_wrap': True, 'valign': 'top','border': 1})
        worksheet.set_column(0, len(df.columns) - 1, None, cell_format)

        # Set row height to 900 pixels
        for row_num in range(len(df) + 1):  # +1 to include header row
            worksheet.set_row(row_num, 50)

        # Add zebra formatting (alternating row colors)
        even_format = workbook.add_format({'text_wrap': True, 'valign': 'top','bg_color': '#D3D3D3','border': 1})  # Light gray background for even rows

        for row_num in range(1, len(df) + 1):  # Start from 1 to skip header row
            if row_num % 2 == 0:
                worksheet.set_row(row_num, 350, even_format)
            else:
                worksheet.set_row(row_num, 350, cell_format, {'bg_color': '#FFFFFF'})


if __name__ == "__main__":
    input_filename = "../apps/data-api/data/manuscripts/manuscriptDescriptionAll.json"
    output_filename = "../apps/data-api/data/manuscripts/test/manuscriptDescriptionAll.xlsx"

    # Ensure the output directory exists
    output_directory = os.path.dirname(output_filename)
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)

    try:
        json_data = read_json(input_filename)
        cleaned_data = clean_json_data(json_data)
        json_to_excel(cleaned_data, output_filename)
        print(f"Excel file successfully generated: {output_filename}")
    except Exception as e:
        print(f"An error occurred: {e}")


Excel file successfully generated: ../apps/data-api/data/manuscripts/test/manuscriptDescriptionAll.xlsx


In [20]:
#merg columns from manuscritpDescriptionAll.jsonimport json

def merge_location_fields(input_file, output_file):
    # Read the JSON data from the input file
    with open(input_file, 'r', encoding='utf-8') as file:
        data = json.load(file)

    for item in data:
        # Merge the specified fields into a new field
        city = item.get('location__city', '')
        library = item.get('location__library', '')
        manuscript_id = item.get('location__manuscript_id', '')
        item['catalogue__location'] = f"City:<br>{city}<br><br> Library:<br>{library} <br><br> Manuscript ID:<br>{manuscript_id}"

        accuracy = item.get('dating__accuracy', '')
        hijri = item.get('dating__hijri_date', '')
        gregorian = item.get('dating__gregorian_date', '')
        item['dating__date'] = f"Accuracy:<br>{accuracy}<br><br> Hijri calendar:<br>{hijri} <br><br> Gregorian calendar:<br>{gregorian}"

        type = item.get('binding__type', '')
        period = item.get('binding__period', '')
        additional_features = item.get('binding__additional_features', '')
        item['binding__features'] = f"Material:<br>{type}<br><br> Period:<br>{period} <br><br> Additional features:<br>{additional_features}"

        script_type = item.get('script__type', '')
        script_hands = item.get('script__hands', '')
        script_execution = item.get('script__execution', '')
        item['script__general'] = f"Type:<br>{script_type}<br><br> Hands:<br>{script_hands} <br><br> Execution:<br>{script_execution}"

        script_size = item.get('script__size', '')
        script_line_spacing = item.get('script__line_spacing', '')

        script_letter_spacing = item.get('script__letter_spacing', '')
        script_stroke_direction = item.get('script__stroke_direction', '')
        script_lower_curves = item.get('script__lower_curves', '')
        script_stroke_thickness = item.get('script__stroke_thickness', '')
        script_baseline = item.get('script__baseline', '')
        script_letter_diacritics = item.get('script__letter_diacritics', '')
        script_vowel_markers = item.get('script__vowel_markers', '')
        item['script__details'] = f"Size:<br>{script_size}<br><br> Line spacing:<br>{script_line_spacing} <br><br> Letter spacing:<br>{script_letter_spacing}<br><br> Stroke direction:<br>{script_stroke_direction}<br><br> Lower curves:<br>{script_lower_curves}<br><br> Stroke thickness:<br>{script_stroke_thickness}<br><br> Baseline:<br>{script_baseline}<br><br> Letter diacritics:<br>{script_letter_diacritics}<br><br> Vowel markers:<br>{script_vowel_markers}"

        orthography_d_dh_shifts = item.get('orthography__d_dh_shifts', '')
        if orthography_d_dh_shifts in ['yes', 'true', 'Yes', 'True', True]:
            orthography_d_dh_shifts = 'Dāl/dāhl shifts'
        elif orthography_d_dh_shifts in ['no', 'false', 'No', 'False', False]:
            orthography_d_dh_shifts = 'No dāl/dāhl shifts'
        item['orthography__d_dh_shifts'] = orthography_d_dh_shifts

        orthography_za_dad_shifts = item.get('orthography__za_dad_shifts', '')
        if orthography_za_dad_shifts in ['yes', 'true', 'Yes', 'True', True]:
            orthography_za_dad_shifts = 'Ḍād/ẓā shifts'
        elif orthography_za_dad_shifts in ['no', 'false', 'No', 'False', False]:
            orthography_za_dad_shifts = 'No ḍād/ẓā shifts'
        item['orthography__za_dad_shifts'] = orthography_za_dad_shifts

        orthography_sin_sad_shifts = item.get('orthography__sin_sad_shifts', '')
        if orthography_sin_sad_shifts in ['yes', 'true', 'Yes', 'True', True]:
            orthography_sin_sad_shifts = 'Sīn/ṣād shifts'
        elif orthography_sin_sad_shifts in ['no', 'false', 'No', 'False', False]:
            orthography_sin_sad_shifts = 'No sīn/ṣād shifts'
        item['orthography__sin_sad_shifts'] = orthography_sin_sad_shifts

        orthography_tha_ta_shifts = item.get('orthography__tha_ta_shifts', '')
        if orthography_tha_ta_shifts in ['yes', 'true', 'Yes', 'True', True]:
            orthography_tha_ta_shifts = 'Tāʾ/thāʾ shifts'
        elif orthography_tha_ta_shifts in ['no', 'false', 'No', 'False', False]:
            orthography_tha_ta_shifts = 'No tāʾ/thāʾ shifts'
        item['orthography__tha_ta_shifts'] = orthography_tha_ta_shifts

        orthography_use_of_hamza = item.get('orthography__use_of_hamza', '')
        item['orthography__sound_shifts'] = f"{orthography_d_dh_shifts}<br><br>{orthography_za_dad_shifts} <br><br>{orthography_sin_sad_shifts}<br><br>{orthography_tha_ta_shifts}<br><br> Use of hamza:<br>{orthography_use_of_hamza}"

        layout_chapter_titles= item.get('layout__chapter_titles', '')
        layout_text_division_symbols = item.get('layout__text_division_symbols', '')
        item['layout__formatting'] = f"Chapter titles:<br>{layout_chapter_titles}<br><br> Text division symbols:<br>{layout_text_division_symbols}"



        # Check for catchwords
        catchwords = item.get('layout__catchwords', '')
        if catchwords in ['yes', 'true', 'Yes', 'True', True]:
            catchwords = 'Catchwords'
        elif catchwords in ['no', 'false', 'No', 'False', False]:
            catchwords = 'No catchwords'
        item['layout__catchwords'] = catchwords

        # Check for frame
        frame = item.get('layout__frame', '')
        if frame in ['yes', 'true', 'Yes', 'True', True]:
            frame = 'Frame'
        elif frame in ['no', 'false', 'No', 'False', False]:
            frame = 'No frame'
        item['layout__frame'] = frame

        line_per_page = item.get('layout__lines_per_page', '')
        item['layout__description'] = f"{catchwords}<br><br>{frame} <br><br> Lines per page:<br>{line_per_page}"

    # Write the modified data back to the output file
    with open(output_file, 'w', encoding='utf-8') as file:
        json.dump(data, file, indent=4, ensure_ascii=False)

# Define the input and output file paths
input_json_file = "manuscriptDescriptionAll.json"
output_json_file = "manuscriptDescriptionAll_modified.json"

# Call the function to merge fields and update the JSON file
merge_location_fields(input_json_file, output_json_file)



In [21]:
import json
import os

def split_json_to_individual_files(input_file, output_folder):
    # Load the data from the JSON file
    with open(input_file, 'r', encoding='utf-8') as file:
        data = json.load(file)

    # Ensure the output folder exists
    os.makedirs(output_folder, exist_ok=True)

    # Write each manuscript's data to a separate JSON file
    for manuscript in data:
        siglum = manuscript.get('siglum__siglum', 'unknown')
        output_path = os.path.join(output_folder, f"{siglum}.json")
        with open(output_path, 'w', encoding='utf-8') as outfile:
            json.dump(manuscript, outfile, indent=4, ensure_ascii=False)

# Define the input file and output folder
input_json_path = "manuscriptDescriptionAll_modified.json"
output_directory = "individual_manuscripts"

# Call the function to split the JSON into individual files
split_json_to_individual_files(input_json_path, output_directory)


In [29]:
import json
import os

def create_grouped_json(input_file, output_file):
    # Load the data from the JSON file
    with open(input_file, 'r', encoding='utf-8') as file:
        data = json.load(file)

    grouped_data = []
    continuum_dict = {}

    # First pass: Collect all manuscripts for each continuum
    for period, manuscripts in data.items():
        for manuscript in manuscripts:
            continuum = manuscript.get('classification', {}).get('continuum', '')
            if continuum:
                if continuum not in continuum_dict:
                    continuum_dict[continuum] = []
                manuscript_name = manuscript.get('manuscript', '').replace('.', '')
                continuum_dict[continuum].append(manuscript_name)

    # Second pass: Create the grouped data with imports
    for period, manuscripts in data.items():
        for manuscript in manuscripts:
            continuum = manuscript.get('classification', {}).get('continuum', '')
            if continuum:
                manuscript_name = manuscript.get('manuscript', '').replace('.', '')
                imports = [ms for ms in continuum_dict[continuum] if ms != manuscript_name]
                grouped_data.append({
                    "imports": imports,
                    "name": manuscript_name,
                    "size": 3931,
                    "century": period,
                    "continuum": continuum
                })

    # Write the grouped data to the output file
    with open(output_file, 'w', encoding='utf-8') as file:
        json.dump(grouped_data, file, indent=4, ensure_ascii=False)

# Define the input file and output file
input_json_path = "individual_manuscripts/new ms_graph.json"
output_json_path = "individual_manuscripts/grouped_manuscripts.json"

# Call the function to create the grouped JSON file
create_grouped_json(input_json_path, output_json_path)


In [31]:
def order_manuscripts_by_century(input_file, output_file):
    # Load the data from the JSON file
    with open(input_file, 'r', encoding='utf-8') as file:
        data = json.load(file)

    # Define a custom sorting key function
    def century_to_int(century):
        # Handle cases like "19th/20th century"
        if '/' in century:
            return int(century.split('/')[0][:-2])  # Use the first century mentioned
        return int(century.split()[0][:-2])  # Extract the number from "XXth century"

    # Sort the manuscripts by century
    sorted_data = sorted(data, key=lambda x: century_to_int(x['century']))

    # Write the sorted data to the output file
    with open(output_file, 'w', encoding='utf-8') as file:
        json.dump(sorted_data, file, indent=4, ensure_ascii=False)

# Define the input file and output file
input_json_path = "individual_manuscripts/grouped_manuscripts.json"
output_json_path = "individual_manuscripts/sorted_grouped_manuscripts.json"

# Call the function to create the sorted JSON file
order_manuscripts_by_century(input_json_path, output_json_path)

print("Manuscripts have been sorted by century and saved to", output_json_path)


Manuscripts have been sorted by century and saved to individual_manuscripts/sorted_grouped_manuscripts.json


In [12]:
import json
import os

def create_manuscript_network_json(input_file, output_file):
    # Load the data from the input JSON file
    with open(input_file, 'r', encoding='utf-8') as file:
        try:
            file_content = file.read()
            cleaned_content = '\n'.join(line for line in file_content.split('\n') if not line.strip().startswith('//'))
            cleaned_content = cleaned_content.replace(',]', ']').replace(',}', '}')
            data = json.loads(cleaned_content)
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON: {e}")
            print(f"Error occurred at line {e.lineno}, column {e.colno}")
            with open(input_file, 'r', encoding='utf-8') as error_file:
                lines = error_file.readlines()
                print(f"Problematic line: {lines[e.lineno - 1]}")
            return

    # Create dictionaries to store manuscripts by continuum, early group and cross copies
    continuum_dict = {}
    early_group_manuscripts = []
    cross_copy_manuscripts = []
    manuscript_names = set()

    # First pass - collect manuscripts by continuum, early group and cross copies
    for manuscript in data:
        manuscript_name = manuscript['name']
        manuscript_names.add(manuscript_name)

        # Check for Early group classification
        classifications = manuscript.get('classification', [])
        if 'Early group' in classifications or 'Early grouü' in classifications:
            early_group_manuscripts.append(manuscript_name)
            if manuscript.get('continuum', '') == '':
                manuscript['continuum'] = 'early group'

        # Check for cross copies
        if manuscript.get('cross_copies', False) is True:
            cross_copy_manuscripts.append(manuscript_name)
            if manuscript.get('continuum', '') == '':
                manuscript['continuum'] = 'cross copy'

        # Group by continuum
        continuum = manuscript.get('continuum', '').lower()
        if continuum:
            if continuum not in continuum_dict:
                continuum_dict[continuum] = []
            continuum_dict[continuum].append(manuscript_name)

    # Create the network structure
    network_data = []
    for manuscript in data:
        imports = []

        # Add manuscripts from same continuum
        continuum = manuscript.get('continuum', '').lower()
        if continuum:
            imports.extend([ms for ms in continuum_dict[continuum]
                          if ms != manuscript['name']
                          and ms.lower() in map(str.lower, manuscript_names)])

        # Add manuscripts with Early group classification
        classifications = manuscript.get('classification', [])
        if 'Early group' in classifications or 'Early Manuscript' in classifications:
            imports.extend([ms for ms in early_group_manuscripts
                          if ms != manuscript['name']
                          and ms not in imports])

        # Add manuscripts with cross_copies true
        if manuscript.get('cross_copies', False) is True:
            imports.extend([ms for ms in cross_copy_manuscripts
                          if ms != manuscript['name']
                          and ms not in imports])

        network_entry = {
            "imports": imports,
            "name": manuscript['name'],
            "size": 3931,
            "century": manuscript['century'],
            "continuum": manuscript.get('continuum', '').lower()
        }
        network_data.append(network_entry)

    # Write the network data to the output file
    with open(output_file, 'w', encoding='utf-8') as file:
        json.dump(network_data, file, indent=4, ensure_ascii=False)

# Define the input file and output file
input_json_path = "individual_manuscripts/graph_new.json"
output_json_path = "individual_manuscripts/network.json"

# Call the function to create the manuscript network JSON file
create_manuscript_network_json(input_json_path, output_json_path)

print("Manuscript network JSON file has been created and saved to", output_json_path)


Manuscript network JSON file has been created and saved to individual_manuscripts/network.json
